# Preparacion del entorno

In [5]:
!pip install pymupdf beautifulsoup4 nbformat

# Paso 2

In [12]:
import os
import fitz  # PyMuPDF
import nbformat
import warnings
from bs4 import BeautifulSoup
from transformers import pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Silenciamos avisos de TensorFlow/Keras para que no molesten
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
warnings.filterwarnings("ignore")

# 2. Funciones de extracción de texto
def extract_text(file_path):
    try:
        if file_path.endswith('.pdf'):
            doc = fitz.open(file_path)
            return "".join([page.get_text() for page in doc])
        elif file_path.endswith('.html'):
            with open(file_path, 'r', encoding='utf-8') as f:
                return BeautifulSoup(f, 'html.parser').get_text()
        elif file_path.endswith('.ipynb'):
            with open(file_path, 'r', encoding='utf-8') as f:
                nb = nbformat.read(f, as_version=4)
                return "\n".join([c.source for c in nb.cells if c.cell_type in ['markdown', 'code']])
    except Exception as e:
        return f"Error en {file_path}: {str(e)}"
    return ""

# 3. Carga del modelo (FORZANDO PyTorch y CPU)
print("⏳ Cargando modelo de inteligencia artificial...")
summarizer = pipeline(
    "summarization", 
    model="facebook/bart-large-cnn", 
    framework="pt", 
    device=-1
)

# 4. Procesamiento por lotes en la carpeta actual
carpeta = "modelo_resumen"  # Tu carpeta 'modelo_resumen'
extensiones = ('.pdf', '.html', '.ipynb')

print(f"🚀 Iniciando resumen de archivos en: {os.getcwd()}\n")

for archivo in os.listdir(carpeta):
    # Saltamos el propio notebook y archivos ocultos
    if archivo.endswith(extensiones) and not archivo.startswith('.') and "Untitled" not in archivo:
        ruta = os.path.join(carpeta, archivo)
        print(f"📄 Procesando: {archivo}...")
        
        texto_completo = extract_text(ruta)
        
        if len(texto_completo) > 100:
            # Dividimos el texto si es muy largo
            splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
            chunks = splitter.split_text(texto_completo)
            
            # Resumimos solo el primer chunk para ir rápido (puedes cambiarlo a bucle si quieres todo)
            resumen = summarizer(chunks[0], max_length=130, min_length=30, do_sample=False)
            
            print(f"✅ RESUMEN: {resumen[0]['summary_text']}\n")
        else:
            print("⚠️ Archivo demasiado corto para resumir.\n")

print("✨ Proceso finalizado.")

⏳ Cargando modelo de inteligencia artificial...
🚀 Iniciando resumen de archivos en: /Users/Jkacholo/iniciacion_NLP_LLM/notebook

📄 Procesando: info.pdf...
✅ RESUMEN: Francisco Espiga Fernández, profesor de Fundamentos Tecnológicos (MU en Business Analytics) Aprenderás obtener valor del dato combinando SQL y Python.

📄 Procesando: modeloresumen.ipynb...
✅ RESUMEN: Pip install pymupdf beautifulsoup4 nbformat. # Preparacion del entorno!pip install tensorFlow/Keras. # Paso 2. # PyMuPDF.

✨ Proceso finalizado.


# Configuración